# Baselines and the ladder

Three rules that fit nothing, then three models of rising flexibility, all scored on the member-grouped folds — and read against the base
rate, because accuracy alone would crown the rule that never calls a wet winter.


## Overview

1. Rung 0: always-dry, always-wet and the Niño-sign rule — two bars and the physical prior.
2. Rungs 1–3: logistic regression, a decision tree, gradient boosting.
3. Confusion matrices, precision, recall and average precision, each against the base rate; why accuracy misleads.
4. The decision threshold: an operating point to trial, not a setting to change silently.

Shared code lives in `lensml.py` beside the chapters (parameters, transforms, ladder, scoring, paired test, register); `show()` prints a
function's source where a chapter introduces it.

The [glossary](glossary.md) defines the terms used here (ladder, rung, incumbent, register, base rate, wet-call share).


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Chapter 3](03-split-by-member.ipynb) | Helpful | the member-grouped folds and `score_splits` |
| scikit-learn User Guide, §3.4 Metrics and scoring | Helpful | precision, recall, confusion matrix |
| Logistic regression, decision trees | Helpful | the idea, not the maths |

- **Time to learn**: about 25 minutes; every cell runs in seconds.
- **System requirements**: the `environment.yml` at the repository root; no account, no local data beyond the committed cache.


## Imports


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from lensml import (
    FEATURES,
    GOLD_CMAP,
    RNG_SEED,
    WET_QUANTILE,
    SignRule,
    cite,
    ladder,
    pooled,
    run_folds,
    setup,
    show,
)
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import FixedThresholdClassifier

In [ ]:
S = setup()  # the cache, the test members, the transforms (chapters 1-2)
train, test = S.train, S.test
X_tr, y_tr = train[FEATURES].values, train["wet"].values
g_tr = train["member_id"].values
print(
    f"train: {S.n_train_members} members / {len(train)} winters   "
    f"test: {test.member_id.nunique()} members / {len(test)} winters   "
    f"N_FOLDS = {S.n_folds}"
)

## The ladder

Unidata compares two algorithms; this cookbook climbs a **ladder of three**, each from scikit-learn (BSD-3-Clause), above a rung of rules.

| Rung | Estimator | What you can read from it | What it costs |
|---|---|---|---|
| 0 | always-dry · always-wet · Niño-sign rule | the two bars, and the physical prior | nothing to fit |
| 1 | `LogisticRegression` | a coefficient per input, in log-odds | assumes a linear boundary |
| 2 | `DecisionTreeClassifier` | the rules themselves | "can be unstable"; overfits without a depth or leaf limit |
| 3 | `GradientBoostingClassifier` | little directly; "an excellent model … for tabular data" | run time; readability; more settings to record |

Rung 0 sets the bars: always-dry on accuracy, always-wet on precision (at the base rate); the sign rule is the prior every rung must beat,
and the first incumbent in chapter 5. Rung 1 because one log-odds coefficient is the first thing a scientist wants to read. Rung 2 because
its rules are readable — does the signal live only in strong events, as NWS Boulder's "moderate to strong" suggests? — under the depth cap
`TREE_SETTINGS` sets. Rung 3 as the most flexible model a one-input table can justify; the table is small, so the guide's own note points
at `GradientBoostingClassifier` rather than the histogram variant.


In [ ]:
show(SignRule, ladder)

## Train

Every estimator takes `random_state=seed` from the declared `RNG_SEED`, and every preprocessing step sits inside a `Pipeline`, so a fit can
be rerun and a transform never sees the scoring side of a fold. The cell also fits rung 1 once on all training members and prints the
one coefficient it learned.


In [ ]:
seed = RNG_SEED
models = ladder(seed)
results = {
    name: run_folds(m, train, seed, S.n_folds) for name, m in models.items()
}
print("trained:", ", ".join(models))
coef = ladder(RNG_SEED)["logistic"].fit(X_tr, y_tr)[-1].coef_[0, 0]
print(
    f"logistic coefficient on the scaled Niño 3.4 anomaly (all training members): "
    f"{coef:+.3f} log-odds per SD"
)

The coefficient is tiny: chapter 2's `standard errors` line already said the straight-line relation is indistinguishable from zero, so
its sign is not evidence for either rule.

## Validate: confusion matrix, precision, recall

Every number below comes from members the fit never saw. The confusion matrix follows scikit-learn's convention — **rows = true class,
columns = predicted**; precision reads down the "predicted wet" column, recall along the "actual wet" row, both for the *wet* class.

The table carries **two estimators, labelled**: `precision_fold` / `recall_fold` are the mean (± SD) of per-fold values, a fold with no wet
call contributing 0; `precision_pooled` / `recall_pooled` are one division over the **summed** matrix, which is what the figure draws.
`called_wet` is the wet-call share. Accuracy, F1 and average precision (AP) stand beside them, because precision or recall alone can be gamed
by predicting one class, and a constant or random score has AP equal to the base rate.


In [ ]:
summary = pd.DataFrame(
    {
        name: dict(
            precision_fold=s["precision"].mean(),
            precision_sd=s["precision"].std(ddof=1),
            recall_fold=s["recall"].mean(),
            recall_sd=s["recall"].std(ddof=1),
            **pooled(cm),
            called_wet=cm[:, 1].sum() / cm.sum(),
            accuracy=s["accuracy"].mean(),
            f1=s["f1"].mean(),
            ap=s["ap"].mean(),
        )
        for name, (s, cm) in results.items()
    }
).T.round(3)
summary["base_rate"] = round(float(y_tr.mean()), 3)
print(
    "nino-sign summed confusion matrix (rows = true, columns = predicted); the "
    "figure below draws the same matrix:"
)
print(
    pd.DataFrame(
        results["nino-sign"][1],
        index=["true dry", "true wet"],
        columns=["pred dry", "pred wet"],
    )
)
summary

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(3.0 * len(results), 3.4))
for ax, (name, (s, cm)) in zip(axes, results.items()):
    ConfusionMatrixDisplay(cm, display_labels=["dry", "wet"]).plot(
        ax=ax, colorbar=False, cmap=GOLD_CMAP
    )
    ax.set_xlim(-0.85, 1.85)
    ax.set_ylim(1.85, -0.85)  # a canvas frame around the matrix
    pr = pooled(cm)
    ax.set_title(
        f"{name}\npooled P={pr['precision_pooled']:.2f} "
        f"R={pr['recall_pooled']:.2f}",
        fontsize=9,
    )
fig.suptitle(
    f"Summed confusion matrices over {S.n_folds} member-grouped folds (rows = "
    "true, columns = predicted); titles = pooled precision/recall",
    fontsize=9,
)
cite(fig, S.access_date)
plt.tight_layout()

### Why accuracy misleads

Find the rung with the highest accuracy in the table, then look at its `called_wet` column. A rung that almost never calls wet scores close
to always-dry's accuracy while catching almost no wet winter: with a base rate near one third, "always dry" is right two times in three. At
the default threshold a probabilistic model says "wet" only when it is more than half sure, and with a weak signal it rarely is. Its recall
then sits near zero — an operating point, not a verdict. The AP column, which reads the ranking at every threshold, says whether a rung
separates wet from dry at all.

```{admonition} Try it: the bars, and the matrix by hand
:class: tip
From the table, say which rung-0 row bars accuracy and which bars precision, and where the sign rule's precision sits against the
always-wet bar. Then, from the `nino-sign` matrix printed above the table, compute its precision and recall by hand. Relabel the four counts
as hits, false alarms, misses and correct rejections, and name POD (= recall) and FAR (= 1 − precision). MET, the Developmental Testbed
Center's verification package, prints the same four counts with rows = forecast and columns = observation, "Yes" first: read every cell off
the axis labels, not off a remembered corner.
```


## The decision threshold is a trial

The decision is a seasonal lean, and the two errors cost differently. A missed wet winter (recall falls) leaves the outlook at equal
chances — a cheap miss. A false alarm (precision falls) leans an outlook wet into a dry winter — the costlier error for an office that will
be held to it. The decision needs both numbers, and chapter 6 counts both for the model it tests.

Lowering the threshold from 0.5 to the designed base rate (`1 - WET_QUANTILE`) trades precision for recall: it moves the operating point
along the same ranking and adds no skill, so AP does not move: the threshold table above prints the same `ap` at both thresholds. The cell below only **shows** that trade-off for logistic regression on the
validation folds. A threshold you keep is recorded as a trial in chapter 5 (`FixedThresholdClassifier`) — and once this table has been
read, that trial is no longer blind.


In [ ]:
# sklearn's default, and the designed base rate
thresholds = [0.5, 1 - WET_QUANTILE]
rows = []
for thr in thresholds:
    m = FixedThresholdClassifier(
        ladder(RNG_SEED)["logistic"],
        threshold=thr,
        response_method="predict_proba",
    )
    s, cm = run_folds(m, train, RNG_SEED, S.n_folds)
    rows.append(
        dict(
            model="logistic",
            threshold=round(thr, 3),
            **pooled(cm),
            f1=s["f1"].mean(),
            ap=s["ap"].mean(),
            predicted_wet=int(cm[:, 1].sum()),
            of=int(cm.sum()),
        )
    )
pd.DataFrame(rows).round(3)

Read the two rows: how many wet calls does each threshold make, what happens to recall, and does AP change?

## Summary

Rung 0 sets two bars and a prior; the ladder adds three models of rising flexibility. Accuracy rewards never calling wet, so every rung is
read on precision, recall and AP against the base rate, with the wet-call share beside them. The decision threshold moves the operating
point, not the ranking — so a threshold is a trial to record, not a default to change.

### What's next?

[Chapter 5](05-refinement-trials.ipynb) climbs the ladder for real: each rung, and each threshold trial, against the incumbent with a
paired test.

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| scikit-learn User Guide: §1.1 Linear models (logistic regression), §1.10 Decision trees, §1.11 Ensembles (gradient-boosted trees). https://scikit-learn.org/stable/ | the three rungs | BSD-3-Clause |
| scikit-learn User Guide, §3.4 Metrics and scoring. https://scikit-learn.org/stable/ | confusion matrix, precision, recall, AP | BSD-3-Clause |
| MET User's Guide, Appendix C: contingency-table statistics (Developmental Testbed Center). https://met.readthedocs.io/en/latest/Users_Guide/appendixC.html | hits, false alarms, POD, FAR | Apache-2.0 |
| NSF Unidata, *Supervised Machine Learning Readiness*. https://github.com/Unidata/supervised-ml-readiness | choose, train, validate | Apache-2.0 |
